In [0]:
%sql
WITH features AS 
(
  SELECT 
      digital_redlining.demographic.fact_geo.geography_id, 
      digital_redlining.mobile_fcc.fact_geo.geography_desc,
      digital_redlining.mobile_fcc.fact_geo.geography_type,
      digital_redlining.mobile_fcc.dim_5g.mobilebb_5g_spd1_area_st_pct,
      digital_redlining.mobile_fcc.dim_5g.mobilebb_5g_spd1_area_iv_pct,
      digital_redlining.mobile_fcc.dim_5g.mobilebb_5g_spd2_area_st_pct,
      digital_redlining.mobile_fcc.dim_5g.mobilebb_5g_spd2_area_iv_pct,
      digital_redlining.demographic.dim_race.percent_white,
      digital_redlining.demographic.dim_race.percent_black,
      digital_redlining.demographic.dim_race.percent_hispanic,
      digital_redlining.demographic.dim_race.percent_asian,
      digital_redlining.demographic.dim_race.percent_hawaiian,
      digital_redlining.demographic.dim_race.percent_native,
      digital_redlining.demographic.dim_race.percent_other
    FROM digital_redlining.mobile_fcc.fact_geo
    INNER JOIN digital_redlining.demographic.fact_geo
      ON digital_redlining.mobile_fcc.fact_geo.geography_id = digital_redlining.demographic.fact_geo.geography_id
    INNER JOIN digital_redlining.mobile_fcc.dim_5g
      ON digital_redlining.mobile_fcc.fact_geo.geography_id = digital_redlining.mobile_fcc.dim_5g.geography_id
    INNER JOIN digital_redlining.demographic.dim_race
      ON digital_redlining.mobile_fcc.fact_geo.geography_id = digital_redlining.demographic.dim_race.geography_id
)

SELECT DISTINCT
  geography_desc,
  geography_type,
  percent_white,
  percent_black,
  mobilebb_5g_spd1_area_st_pct,
  mobilebb_5g_spd1_area_iv_pct,
  mobilebb_5g_spd2_area_st_pct,
  mobilebb_5g_spd2_area_iv_pct
FROM features
WHERE geography_type = 'County'
ORDER BY percent_white DESC
LIMIT 10;


-- SELECT 
--   geography_type,
--   AVG(mobilebb_5g_spd1_area_st_pct) AS avg_spd1_area_st_pct,
--   AVG(mobilebb_5g_spd1_area_iv_pct) AS avg_spd1_area_iv_pct,
--   AVG(mobilebb_5g_spd2_area_st_pct) AS avg_spd2_area_st_pct,
--   AVG(mobilebb_5g_spd2_area_iv_pct) AS avg_spd2_area_iv_pct,
--   percent_white
-- FROM features
-- GROUP BY geography_type, percent_white;

In [0]:
import requests


endpoint = "https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=county"

response = requests.get(endpoint)

if response.status_code == 200:
    json_data = response.json()
else:
    raise Exception(f"Request failed with status code {response.status_code}")

import pandas as pd

df = pd.DataFrame(json_data[1:], columns=json_data[0])
df = df.rename(columns={
    "NAME": "County Name",
    "P1_001N": "Total Population",
    "P1_003N": "White Population",
    "P1_004N": "Black or African American Population",
    "P1_005N": "American Indian and Alaska Native Population",
    "P1_006N": "Asian Population",
    "P1_007N": "Native Hawaiian and Other Pacific Islander Population",
    "P1_008N": "Some Other Race Population",
    "P1_009N": "Two or More Races Population",
    "P2_001N": "Total Housing Units",
    "P2_002N": "Occupied Housing Units",
    "H1_001N": "Total Households",
    "H1_002N": "Family Households",
    "state": "State Code",
    "county": "County Code"
})

display(df)

In [0]:
import pyspark.sql.functions as F
import requests

endpoint = "https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=county"

response = requests.get(endpoint)
if response.status_code == 200:
    json_data = response.json()
else:
    raise Exception(f"Request failed with status code {response.status_code}")

columns = json_data[0]
data = json_data[1:]

spark_df = spark.createDataFrame(data, columns)
spark_df = spark_df.withColumn('geo_level', F.lit('county'))
display(spark_df)

In [0]:
endpoint = "https://api.census.gov/data/2020/dec/pl?get=GEO_ID,NAME,P1_001N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N,P1_009N,P2_001N,P2_002N,H1_001N,H1_002N&for=state"

response = requests.get(endpoint)
if response.status_code == 200:
    json_data = response.json()
else:
    raise Exception(f"Request failed with status code {response.status_code}")

columns = json_data[0]
data = json_data[1:]

spark_df = spark.createDataFrame(data, columns)
display(spark_df)